# 04 — Évolution des topics de séance en séance (sankey)

Objectif : voir de quoi parlent les orateurs à chaque séance et comment les sujets se prolongent, se scindent ou s'éteignent d'une séance à l'autre.

Méthode :
- **une séance = une date** : les prises de parole de la Chambre et du Sénat du même jour sont réunies ;
- même corpus et même sac de mots que `03_topics_lda` (vocabulaire commun à toutes les dates, pour pouvoir comparer les topics) ;
- **une LDA par date**, à `K = n / DOCS_PER_TOPIC` topics (borné à [`K_MIN`, `K_MAX`]) ; chaque prise de parole est rattachée à son topic dominant, et un topic pèse le **nombre de prises de parole** qui lui sont rattachées ; les dates de moins de `MIN_DOCS` prises de parole sont écartées ;
- **liens** entre topics de deux dates consécutives : similarité cosinus de leurs distributions de mots ; seules les paires au-dessus de `SIM_MIN` sont reliées ;
- **largeur d'un ruban** A → B : la part de A attribuée à B au prorata des similarités, bornée par la part de B attribuée à A — un ruban ne transporte jamais plus de prises de parole que n'en comptent A et B, si bien que la hauteur de chaque topic est exactement son nombre de prises de parole ; la part d'un topic qui ne se prolonge pas (sujet clos, ou propre à cette séance) reste sans ruban ;
- **couleur** : un topic hérite de la « lignée » de son principal prédécesseur s'il en reçoit au moins 30 % de ses prises de parole ; les huit lignées de sujets les plus fournies (sur au moins deux dates, hors procédure) sont colorées, les autres en gris.

Sortie : `output/sankey/sankey_topics.html`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import LatentDirichletAllocation

from debats import OUTPUT_DIR
from debats.chunks import load_chunks, strip_speaker
from debats.topics import fold_plurals, make_vectorizer, name_topics, short_sections, top_words

pd.set_option("display.max_colwidth", 120)

MIN_WORDS = 40  # longueur minimale d'une prise de parole (sans le nom de l'orateur)
MIN_DOCS = 20  # prises de parole minimum pour modéliser une date
DOCS_PER_TOPIC = 40
K_MIN, K_MAX = 2, 8
SIM_MIN = 0.4  # similarité cosinus minimale pour relier deux topics
LINEAGE_SHARE = 0.3  # part minimale reçue du prédécesseur pour en hériter la couleur
HIDE_PROCEDURE = False  # masquer les topics de procédure (dépôts, renvois, ordre du jour)
SEED = 0

## Corpus

In [ ]:
chunks = load_chunks()
chunks["body"] = chunks.text.map(strip_speaker)
chunks["n_words"] = chunks.body.str.split().str.len()
chunks["rubrique"] = short_sections(chunks.section)

corpus = chunks[chunks.n_words >= MIN_WORDS].reset_index(drop=True)
print(f"{len(corpus)} prises de parole d'au moins {MIN_WORDS} mots sur {len(chunks)}")

In [ ]:
vectorizer = make_vectorizer()
X, vocab = fold_plurals(vectorizer.fit_transform(corpus.body), vectorizer.get_feature_names_out())
print(X.shape, "(prises de parole × mots)")
corpus.groupby(["date", "chamber"]).size().unstack(fill_value=0).assign(total=lambda d: d.sum(axis=1)).T

## Topics par date

Chaque LDA ne voit que les mots présents dans au moins deux prises de parole de la date ; les distributions de mots sont replacées dans le vocabulaire commun pour la comparaison entre dates.

In [ ]:
def date_topics(date: pd.Timestamp, rows: np.ndarray) -> pd.DataFrame:
    cols = np.flatnonzero(np.asarray((X[rows] > 0).sum(axis=0)).ravel() >= 2)
    Xd = X[rows][:, cols]
    k = int(np.clip(round(len(rows) / DOCS_PER_TOPIC), K_MIN, K_MAX))
    lda = LatentDirichletAllocation(k, learning_method="batch", max_iter=50, random_state=SEED).fit(Xd)
    doc_topic = lda.transform(Xd)
    keywords = top_words(lda.components_, vocab[cols], np.asarray(Xd.sum(axis=0)).ravel())
    topics = name_topics(keywords, doc_topic, corpus.rubrique.iloc[rows])
    phi = np.zeros((k, len(vocab)))
    phi[:, cols] = lda.components_ / lda.components_.sum(axis=1, keepdims=True)

    dominant = doc_topic.argmax(axis=1)
    chamber = corpus.chamber.to_numpy()[rows]
    topics["id"] = [f"{date:%Y-%m-%d}#{t}" for t in topics.index]
    topics["date"] = date
    topics["phi"] = list(phi)
    topics["chambre"] = [int(((dominant == t) & (chamber == "chambre")).sum()) for t in topics.index]
    topics["senat"] = [int(((dominant == t) & (chamber == "senat")).sum()) for t in topics.index]
    corpus.loc[rows, "topic_id"] = topics.id.to_numpy()[dominant]
    return topics[topics.n_docs > 0]


corpus["topic_id"] = None
groups = corpus.groupby("date").indices
skipped = {f"{d:%d/%m}": len(rows) for d, rows in groups.items() if len(rows) < MIN_DOCS}
print("dates écartées (trop peu de prises de parole) :", skipped)
topics = pd.concat([date_topics(d, rows) for d, rows in groups.items() if len(rows) >= MIN_DOCS]).set_index("id")
if HIDE_PROCEDURE:
    topics = topics[~topics.procedure]
dates = sorted(topics.date.unique())
print(len(topics), "topics sur", len(dates), "dates")
topics[["date", "name", "n_docs", "chambre", "senat", "keywords"]]

## Liens entre dates consécutives

Distribution des similarités entre topics de dates consécutives : la plupart des paires partagent seulement le vocabulaire commun des débats ; `SIM_MIN` ne garde que la queue de la distribution.

In [ ]:
def cosine(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A @ B.T


links, similarities = [], []
for a, b in zip(dates, dates[1:]):
    A, B = topics[topics.date == a], topics[topics.date == b]
    S = cosine(np.vstack(A.phi), np.vstack(B.phi))
    similarities.extend(S.ravel())
    W = np.where(S >= SIM_MIN, S, 0.0)
    with np.errstate(invalid="ignore"):
        # part de A attribuée à B (au prorata des similarités), et réciproquement
        out_share = np.nan_to_num(A.n_docs.to_numpy()[:, None] * W / W.sum(axis=1, keepdims=True))
        in_share = np.nan_to_num(B.n_docs.to_numpy()[None, :] * W / W.sum(axis=0, keepdims=True))
    flow = np.minimum(out_share, in_share)
    for i, j in zip(*np.nonzero(flow)):
        links.append({"source": A.index[i], "target": B.index[j], "value": flow[i, j], "sim": S[i, j]})
links = pd.DataFrame(links)

fig = px.histogram(x=similarities, nbins=50, labels={"x": "similarité cosinus"}, height=300)
fig.add_vline(SIM_MIN, line_dash="dash")
fig.update_layout(template="plotly_white", yaxis_title="paires de topics", margin={"l": 10, "r": 10, "t": 30, "b": 10})
fig.show()
print(f"{len(links)} liens ; {links.value.sum():.0f} prises de parole sur {topics.n_docs.sum()} se prolongent d'une date à la suivante")

## Lignées et couleurs

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
OTHER = "#a8a7a2"

lineage: dict[str, str] = {}
for date in dates:
    incoming = links[links.target.isin(topics.index[topics.date == date])].sort_values("value", ascending=False)
    taken = set()  # une lignée ne se prolonge que dans un topic par date
    for link in incoming.itertuples():
        heir = lineage[link.source]
        if link.target in lineage or heir in taken or link.value < LINEAGE_SHARE * topics.n_docs[link.target]:
            continue
        lineage[link.target] = heir
        taken.add(heir)
    for topic in topics.index[topics.date == date]:
        lineage.setdefault(topic, topic)
topics["lineage"] = pd.Series(lineage)

lineages = topics.groupby("lineage").agg(total=("n_docs", "sum"), dates=("date", "nunique"))
lineages["name"] = topics.sort_values("n_docs").groupby("lineage").name.last()
lineages["procedure"] = topics.sort_values("n_docs").groupby("lineage").procedure.last()
colored = lineages[(lineages.dates >= 2) & ~lineages.procedure].nlargest(len(PALETTE), "total").index
lineages["color"] = OTHER
lineages.loc[colored, "color"] = PALETTE[: len(colored)]
topics["color"] = topics.lineage.map(lineages.color)
lineages.loc[colored]

## Sankey

Les dates sont en abscisse (une colonne par date, dans l'ordre), les topics empilés dans chaque colonne, l'axe vertical est en nombre de prises de parole. Les topics d'une colonne sont ordonnés selon la position moyenne de leurs prédécesseurs pour limiter les croisements.

Survol : un topic donne sa date, son nom, son nombre de prises de parole (Chambre / Sénat), ses mots-clés et ses rubriques ; un ruban, le nombre de prises de parole qui se prolongent et la similarité. Un clic sur une lignée de la légende la masque.

In [ ]:
NODE_WIDTH = 0.14  # en fraction de l'écart entre deux dates
gap = 0.04 * topics.groupby("date").n_docs.sum().max()
height = max(g.n_docs.sum() + gap * (len(g) - 1) for _, g in topics.groupby("date"))

# Position verticale des topics, colonne par colonne
topics["x"] = topics.date.map({d: i for i, d in enumerate(dates)})
topics["y0"] = np.nan
for date in dates:
    column = topics[topics.date == date]
    incoming = links[links.target.isin(column.index)].assign(
        center=lambda d: d.source.map(topics.y0 + topics.n_docs / 2)
    )
    barycenter = incoming.groupby("target").apply(lambda d: np.average(d.center, weights=d.value), include_groups=False)
    order = sorted(column.index, key=lambda t: (t not in barycenter.index, barycenter.get(t, 0), -column.n_docs[t]))
    y = (height - column.n_docs.sum() - gap * (len(column) - 1)) / 2
    for topic in order:
        topics.loc[topic, "y0"] = y
        y += column.n_docs[topic] + gap

# Emplacement des rubans sur leurs topics : empilés dans l'ordre vertical de l'autre extrémité
links["source_y"] = links.groupby("source", group_keys=False).apply(
    lambda d: topics.y0[d.name] + d.value.reindex(d.target.map(topics.y0).sort_values().index).cumsum().shift(fill_value=0),
    include_groups=False,
)
links["target_y"] = links.groupby("target", group_keys=False).apply(
    lambda d: topics.y0[d.name] + d.value.reindex(d.source.map(topics.y0).sort_values().index).cumsum().shift(fill_value=0),
    include_groups=False,
)
links["color"] = links.source.map(topics.color)

In [ ]:
def ribbon(x0: float, x1: float, y0: float, y1: float, width: float, n: int = 24) -> tuple[list, list]:
    # Polygone d'un ruban : deux courbes en S (smoothstep) de largeur constante, fermé par None.
    s = np.linspace(0, 1, n)
    ease = s * s * (3 - 2 * s)
    xs = x0 + (x1 - x0) * s
    top = y0 + (y1 - y0) * ease
    return [*xs, *xs[::-1], None], [*top, *(top + width)[::-1], None]


def rgba(color: str, alpha: float) -> str:
    r, g, b = (int(color[i : i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r},{g},{b},{alpha})"


def hover_list(items, n: int) -> str:
    return ", ".join(items[:n])


fig = go.Figure()
for color in [*PALETTE, OTHER]:
    group = links[links.color == color]
    nodes = topics[topics.color == color]
    if nodes.empty:
        continue
    name = lineages.name[lineages.color == color].iloc[0] if color != OTHER else "autres sujets et procédure"
    xs, ys = [], []
    for link in group.itertuples():
        rx, ry = ribbon(
            topics.x[link.source] + NODE_WIDTH / 2, topics.x[link.target] - NODE_WIDTH / 2, link.source_y, link.target_y, link.value
        )
        xs += rx
        ys += ry
    fig.add_scatter(
        x=xs, y=ys, fill="toself", mode="lines", line_width=0, fillcolor=rgba(color, 0.35),
        hoverinfo="skip", legendgroup=color, showlegend=False,
    )
    fig.add_bar(
        x=nodes.x, y=nodes.n_docs, base=nodes.y0, width=NODE_WIDTH, marker_color=color, marker_line_width=0,
        name=name if len(name) <= 60 else name[:59] + "…", legendgroup=color,
        customdata=np.c_[
            nodes.name, nodes.date.dt.strftime("%d/%m/%Y"), nodes.n_docs, nodes.chambre, nodes.senat,
            nodes.keywords.map(lambda k: hover_list(k, 8)),
            nodes.top_sections.map(lambda s: "<br>".join(f"· {r} ({n})" for r, n in s.items())),
        ],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>%{customdata[1]} · %{customdata[2]} prises de parole"
            " (Chambre %{customdata[3]}, Sénat %{customdata[4]})<br><i>%{customdata[5]}</i><br>%{customdata[6]}<extra></extra>"
        ),
    )

# Survol des rubans : un point invisible au milieu de chaque ruban
fig.add_scatter(
    x=(links.source.map(topics.x) + links.target.map(topics.x)) / 2,
    y=(links.source_y + links.target_y + links.value) / 2,
    mode="markers", marker={"size": 14, "opacity": 0}, showlegend=False,
    customdata=np.c_[links.source.map(topics.name), links.target.map(topics.name), links.value.round(), links.sim.round(2)],
    hovertemplate="%{customdata[0]}<br>→ %{customdata[1]}<br><b>%{customdata[2]}</b> prises de parole · similarité %{customdata[3]}<extra></extra>",
)

per_date = topics.groupby("date")[["chambre", "senat"]].sum()
ticks = [
    f"{d:%d/%m}<br>{'C' if per_date.chambre[d] else ''}{'+' if per_date.chambre[d] and per_date.senat[d] else ''}{'S' if per_date.senat[d] else ''}"
    for d in dates
]
fig.update_layout(
    template="plotly_white",
    title="Topics des prises de parole de séance en séance — Chambre (C) et Sénat (S), janvier–mars 1934",
    barmode="overlay",
    height=900,
    hoverlabel_align="left",
    legend={"title": "Lignées de topics", "orientation": "h", "y": -0.12, "font_size": 11},
    xaxis={"tickmode": "array", "tickvals": list(range(len(dates))), "ticktext": ticks, "showgrid": False, "zeroline": False, "range": [-0.5, len(dates) - 0.5]},
    yaxis={"autorange": "reversed", "visible": False},
    margin={"l": 20, "r": 20, "t": 60, "b": 20},
)
fig.show()

## Écriture

In [ ]:
out_dir = OUTPUT_DIR / "sankey"
out_dir.mkdir(parents=True, exist_ok=True)
fig.write_html(out_dir / "sankey_topics.html", include_plotlyjs="cdn")
topics.assign(keywords=topics.keywords.str.join(", ")).drop(columns=["phi", "top_sections"]).to_csv(out_dir / "date_topics.csv")
links.to_csv(out_dir / "links.csv", index=False)
print("→", out_dir)